<a href="https://colab.research.google.com/github/abeecodes/Ecommerce_ChatBot/blob/main/ECOMMERCE_FAQ_BOT_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Commerce FAQ Bot
**Domain:** Online retail customer support  
**User:** Online shoppers asking about products, shipping, returns, and policies  
**Setup required (Colab Secrets):**
- `GROQ_API_KEY` — required for the LLM
- `NGROK_AUTH_TOKEN` — required to expose the Streamlit UI

Run all cells top-to-bottom. The last cell prints your public URL.

## Part 1 — Install & Imports

In [1]:
!pip uninstall -y google-adk opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp-proto-http -q

!pip install -q \
    chromadb \
    sentence-transformers \
    langchain-community \
    langchain-groq \
    langgraph \
    streamlit \
    pyngrok \
    langchain-huggingface \
    ragas

print('All packages installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.9/515.9 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 

In [2]:
import os, re, uuid, datetime, subprocess, time
from typing import TypedDict, List, Optional
from google.colab import userdata
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

def _secret(key):
    try:
        return userdata.get(key)
    except Exception:
        return None

GROQ_API_KEY  = _secret('GROQ_API_KEY')
NGROK_TOKEN   = _secret('NGROK_AUTH_TOKEN')

if GROQ_API_KEY:
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
    print('GROQ_API_KEY loaded.')
else:
    print('WARNING: GROQ_API_KEY not found. Add it to Colab Secrets.')

if NGROK_TOKEN:
    print('NGROK_AUTH_TOKEN loaded.')
else:
    print('WARNING: NGROK_AUTH_TOKEN not found. Add it to Colab Secrets.')

GROQ_API_KEY loaded.
NGROK_AUTH_TOKEN loaded.


## Part 1 — Knowledge Base

In [3]:
DOCUMENTS = [

    #PRODUCTS

    {
        'id': 'doc_001',
        'category': 'Electronics > Audio',
        'topic': 'Wireless Headphones',
        'sku': 'WH-300',
        'price': 1999,
        'discount': '10%',
        'stock_status': 'In Stock',
        'tags': ['bluetooth', 'anc', 'headphones', 'audio'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax Wireless Headphones (Model WH-300) use Bluetooth 5.3 for a stable, '
            'low-latency connection up to 10 metres. Battery life is 20 hours on a single charge, '
            'with a 2-hour fast-charge via USB-C. The headphones include active noise cancellation (ANC), '
            'a built-in microphone for calls, and 40mm drivers for rich bass. Weight is 250g. '
            'They are foldable and come with a hard carrying case. Available colours: Black, White, Navy Blue. '
            'Price: Rs 1,999. Currently available at a 10% discount. Covered by a 1-year manufacturer warranty '
            'against hardware defects. Compatible with iOS, Android, Windows, and macOS. Not waterproof.'
        )
    },

    {
        'id': 'doc_002',
        'category': 'Electronics > Wearables',
        'topic': 'Smart Watch',
        'sku': 'SW-200',
        'price': 2999,
        'discount': '10%',
        'stock_status': 'In Stock',
        'tags': ['fitness', 'watch', 'gps', 'health'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax Smart Watch (Model SW-200) tracks steps, calories, sleep, heart rate, '
            'and blood oxygen (SpO2). It has a 1.4-inch AMOLED display with always-on mode. '
            'Battery life is 7 days in normal mode, 3 days with always-on display enabled. '
            'It is IP68 waterproof — safe for swimming up to 50 metres. Compatible with iOS 12+ and Android 8+. '
            'Includes 20+ workout modes (running, cycling, yoga, swimming). Built-in GPS for outdoor runs. '
            'Supports notifications for calls, SMS, and apps. Strap is interchangeable (22mm standard). '
            'Price: Rs 2,999. Currently available at a 10% discount. 1-year warranty.'
        )
    },

    {
        'id': 'doc_003',
        'category': 'Electronics > Accessories',
        'topic': 'Mechanical Keyboard',
        'sku': 'KB-75',
        'price': 3499,
        'discount': '10%',
        'stock_status': 'In Stock',
        'tags': ['keyboard', 'gaming', 'rgb'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax Mechanical Keyboard (Model KB-75) is a tenkeyless (TKL) wired keyboard with '
            'Cherry MX Red switches — linear, quiet, and ideal for both typing and gaming. '
            'It has per-key RGB backlighting with 15 preset lighting modes. The aluminium top plate and '
            'PBT double-shot keycaps make it durable and resistant to shine. N-key rollover (NKRO) ensures '
            'every keypress is registered simultaneously. Connection: USB-A braided cable (1.8m). '
            'Layout: US ANSI. Compatible with Windows 10/11 and macOS. No driver software required. '
            'Price: Rs 3,499. Currently available at a 10% discount. 2-year warranty.'
        )
    },

    {
        'id': 'doc_004',
        'category': 'Electronics > Audio',
        'topic': 'Portable Bluetooth Speaker',
        'sku': 'BS-50',
        'price': 2499,
        'discount': None,
        'stock_status': 'In Stock',
        'tags': ['speaker', 'waterproof', 'bluetooth'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax Blast Speaker (Model BS-50) delivers 20W stereo sound with dual 10W drivers '
            'and a passive radiator for deep bass. Battery lasts 12 hours at 70% volume. '
            'It is IPX7 waterproof — fully submersible up to 1 metre for 30 minutes. '
            'Bluetooth 5.0 range is up to 15 metres. Also supports wired AUX input (3.5mm). '
            'Has a built-in microphone for speakerphone calls. Charges in 3 hours via USB-C. '
            'Dimensions: 18cm x 7cm. Weight: 600g. Colours: Red, Black, Forest Green. '
            'Price: Rs 2,499. 1-year warranty.'
        )
    },

    {
        'id': 'doc_005',
        'category': 'Electronics > Charging',
        'topic': 'USB-C Fast Charger',
        'sku': 'FC-65',
        'price': 1299,
        'discount': '5%',
        'stock_status': 'In Stock',
        'tags': ['charger', 'gan', 'fast-charge'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax 65W USB-C GaN Charger (Model FC-65) supports USB Power Delivery 3.0 '
            'and charges laptops, tablets, and phones. It has two ports: one USB-C (65W max) and '
            'one USB-A (18W max, Quick Charge 3.0). The GaN (Gallium Nitride) technology makes it '
            '40% smaller than traditional chargers of the same wattage. '
            'Charges an iPhone 15 to 50% in 30 minutes. Charges a MacBook Air in under 2 hours. '
            'Universal voltage (100-240V) — suitable for international travel. '
            'Cable not included. Price: Rs 1,299. 1-year warranty. Colour: White.' )
    },

    {
        'id': 'doc_006',
        'category': 'Accessories',
        'topic': 'Laptop Stand',
        'sku': 'LS-20',
        'price': 899,
        'discount': None,
        'stock_status': 'In Stock',
        'tags': ['ergonomic', 'stand'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax ErgoRise Laptop Stand (Model LS-20) is made from anodised aluminium and '
            'supports laptops from 10 to 16 inches weighing up to 10kg. '
            'Height is adjustable across 6 levels (10cm to 20cm) and the angle can be set from 15 to 45 degrees. '
            'Folds flat in seconds for portability — packed size is 28cm x 22cm x 1.5cm, weight 500g. '
            'Silicone pads grip the desk and protect the laptop from scratches. '
            'Does not include a USB hub. Compatible with all brands (Apple, Dell, HP, Lenovo, Asus, etc.). '
            'Price: Rs 899. 1-year warranty. Available in Silver and Space Grey.'
                 )
    },

    {
        'id': 'doc_013',
        'category': 'Electronics > Power',
        'topic': 'Power Bank',
        'sku': 'PB-20000',
        'price': 1499,
        'discount': None,
        'stock_status': 'In Stock',
        'tags': ['battery', 'portable'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax Power Bank (Model PB-20000) has a 20,000mAh capacity and supports '
        '18W fast charging via USB-C Power Delivery and Quick Charge 3.0. '
        'It includes 2 USB-A ports and 1 USB-C port (input/output). '
        'Can charge a smartphone up to 4-5 times depending on battery size. '
        'LED indicators display remaining battery level. '
        'Recharges fully in 6-7 hours with a 18W charger. '
        'Built-in protections against overcharge, over-discharge, and short circuit. '
        'Weight: 450g. Price: Rs 1,499. 1-year warranty. Colour: Black.'
        )
    },

    {
        'id': 'doc_014',
        'category': 'Electronics > Audio',
        'topic': 'True Wireless Earbuds',
        'sku': 'TWS-100',
        'price': 1799,
        'discount': None,
        'stock_status': 'In Stock',
        'tags': ['earbuds', 'tws'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax True Wireless Earbuds (Model TWS-100) feature Bluetooth 5.2 connectivity '
        'with auto-pairing and touch controls. Battery life is 5 hours per charge, '
        'with an additional 20 hours using the charging case. '
        'Supports ENC (Environmental Noise Cancellation) for clearer calls. '
        'IPX5 water resistance protects against sweat and splashes. '
        'Charging case uses USB-C and supports fast charging (10 min = 1 hour playback). '
        'Includes 3 sizes of silicone ear tips. Price: Rs 1,799. 1-year warranty.'
        )
    },

    {
        'id': 'doc_015',
        'category': 'Electronics > Gaming',
        'topic': 'Gaming Mouse',
        'sku': 'GM-80',
        'price': 1299,
        'discount': None,
        'stock_status': 'In Stock',
        'tags': ['gaming', 'mouse'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax Gaming Mouse (Model GM-80) features a 16,000 DPI optical sensor with '
        'adjustable sensitivity levels. Includes 6 programmable buttons and RGB lighting '
        'with customizable effects. Polling rate is 1000Hz for ultra-fast response. '
        'Ergonomic design suitable for right-handed users. '
        'Connection via USB-A braided cable (1.6m). Compatible with Windows and macOS. '
        'Software required for customization. Weight: 120g. Price: Rs 1,299. 1-year warranty.'
        )
    },

    {
        'id': 'doc_016',
        'category': 'Electronics > Storage',
        'topic': 'External SSD',
        'sku': 'SSD-1TB',
        'price': 6999,
        'discount': None,
        'stock_status': 'In Stock',
        'tags': ['ssd', 'storage'],
        'last_updated': '2026-04-23',
        'text': (
            'The ShopMax External SSD (Model SSD-1TB) offers 1TB storage with read speeds up to 1050 MB/s '
        'and write speeds up to 1000 MB/s. Uses USB 3.2 Gen 2 interface with USB-C connectivity. '
        'Compact and shock-resistant design with aluminium casing for heat dissipation. '
        'Compatible with Windows, macOS, Android (OTG supported), and gaming consoles. '
        'Includes USB-C to C and USB-C to A cables. '
        'Weight: 90g. Price: Rs 6,999. 3-year warranty.'
        )
    },


    #POLICIES

    {
        'id': 'doc_007',
        'category': 'Policy',
        'topic': 'Return Policy',
        'last_updated': '2026-04-23',
        'text': ('Customers may return any product within 7 days of delivery for a full refund, '
            'provided the item is unused, in its original packaging, and all accessories and manuals are included. '
            'To initiate a return, log in to your ShopMax account, go to Orders, select the item, and click Return. '
            'A pickup will be arranged within 1-2 business days at no cost to the customer. '
            'Items that are damaged due to customer misuse, have missing parts, or are returned after 7 days '
            'are not eligible for a refund. '
            'Opened software, downloaded digital products, and personalised/custom items cannot be returned. '
            'Refunds are processed within 5-7 business days of the return being received and verified.'
        )
    },

    {
        'id': 'doc_008',
        'category': 'Policy',
        'topic': 'Shipping Policy',
        'last_updated': '2026-04-23',
        'text': (
            'ShopMax ships to all major cities and towns across India. '
            'Standard delivery takes 3-5 business days and is free for orders above Rs 499. '
             'For orders below Rs 499, a flat shipping fee of Rs 49 applies. '
            'Express delivery (1-2 business days) costs Rs 99 regardless of order value. '
            'Same-day delivery is available in Bangalore, Mumbai, Delhi, Hyderabad, and Chennai for orders placed before 11am. '
            'Same-day delivery costs Rs 149. '
            'Orders are dispatched Monday to Saturday (excluding public holidays). '
            'A tracking link is sent via SMS and email once the order ships. '
            'International shipping is not currently available.'
            )
    },

    {
        'id': 'doc_009',
        'category': 'Policy',
        'topic': 'Refund Policy',
        'last_updated': '2026-04-23',
        'text': (
            'Once a returned item is received and verified by our warehouse team, the refund is processed '
            'within 5-7 business days. '
            'For orders paid by credit or debit card, the refund appears on the original card. '
            'For UPI and net banking payments, the refund is credited to the original payment account. '
            'Cash on Delivery orders are refunded via bank transfer — customers must provide their bank account number and IFSC code. '
            'ShopMax Wallet refunds (store credit) are processed within 24 hours. '
            'Customers receive an email confirmation when the refund is initiated. '
            'If the refund is not received within 7 business days, contact support with your order ID.'
            )
    },

    {
        'id': 'doc_010',
        'category': 'Policy',
        'topic': 'Warranty Policy',
        'last_updated': '2026-04-23',
        'text': (
            'All ShopMax products include a manufacturer warranty covering hardware defects. '
            'Most products carry a 1-year warranty; the Mechanical Keyboard (Model KB-75) carries a 2-year warranty. '
            'Warranty claims must be submitted within the warranty period by contacting support at support@shopmax.in or calling 1800-123-4567 (toll-free). '
            'The warranty covers: dead-on-arrival (DOA) units, manufacturing defects, and component failure under normal use. '
            'The warranty does NOT cover: physical damage, water damage (unless the product is rated waterproof), '
            'damage from incorrect voltage, and damage from unauthorised repair. '
            'Proof of purchase (order ID or invoice) is required for all warranty claims. '
            'Approved warranty claims result in a free repair or replacement at ShopMax discretion.'
            )
    },

    {
        'id': 'doc_011',
        'category': 'Policy',
        'topic': 'Payment Methods',
        'last_updated': '2026-04-23',
        'text': (
            'ShopMax accepts the following payment methods: '
            'Credit cards (Visa, Mastercard, Amex, RuPay), debit cards (all major banks), '
            'UPI (Google Pay, PhonePe, Paytm, BHIM, and any UPI app), '
            'net banking (50+ banks supported), Cash on Delivery (COD) for orders up to Rs 10,000, '
            'and ShopMax Wallet (store credit). '
            'EMI is available on orders above Rs 3,000 via eligible credit cards (3, 6, 9, and 12-month tenures). '
            'Buy Now Pay Later (BNPL) is available through ZestMoney and LazyPay for eligible customers. '
            'All transactions are secured with 256-bit SSL encryption. '
            'Prices shown are inclusive of GST (18% for electronics).'
            )
    },

    {
        'id': 'doc_012',
        'category': 'Policy',
        'topic': 'Order Tracking and Cancellation',
        'last_updated': '2026-04-23',
        'text': (
            'Once an order is placed, a confirmation email and SMS are sent immediately. '
            'After dispatch, a tracking link from our courier partner (Delhivery, Ekart, or BlueDart) is shared via SMS and email. '
            'To track your order, visit shopmax.in/track or use the ShopMax app and enter your order ID. '
            'Orders can be cancelled for free within 12 hours of placement if not yet dispatched. '
            'To cancel, go to My Orders in your account and click Cancel Order. '
            'Orders that are already dispatched cannot be cancelled — you must wait for delivery and then raise a return request. '
            'Refunds for cancelled pre-dispatch orders are processed within 24 hours. '
            'For any issues with tracking or cancellation, contact support@shopmax.in or call 1800-123-4567.'
            )
    },
    {
    'id': 'doc_017',
    'topic': 'Exchange Policy',
    'text': (
        'ShopMax allows product exchanges within 7 days of delivery for items that are defective '
        'or damaged on arrival. Customers must request an exchange through their account under Orders. '
        'The replacement will be shipped after the original item is picked up and inspected. '
        'Exchanges are subject to stock availability. '
        'If the product is unavailable, a full refund will be issued instead. '
        'Products must be returned with original packaging and all accessories.'
    )
},
{
    'id': 'doc_018',
    'topic': 'Account and Security Policy',
    'text': (
        'Customers are responsible for maintaining the confidentiality of their ShopMax account credentials. '
        'ShopMax uses secure login systems and 256-bit SSL encryption to protect user data. '
        'Users should not share OTPs, passwords, or personal information with anyone. '
        'Any suspicious activity should be reported immediately to support@shopmax.in. '
        'ShopMax is not liable for unauthorized access due to user negligence.'
    )
},
{
    'id': 'doc_019',
    'topic': 'Privacy Policy',
    'text': (
        'ShopMax collects customer data such as name, contact details, and payment information '
        'to process orders and improve services. '
        'Data is stored securely and is not sold to third parties. '
        'Information may be shared with logistics partners for delivery purposes. '
        'Customers can request data deletion by contacting support. '
        'Cookies are used to enhance browsing experience and personalize recommendations.'
    )
},
{
    'id': 'doc_020',
    'topic': 'Customer Support Policy',
    'text': (
        'ShopMax customer support is available Monday to Saturday, 9am to 7pm IST. '
        'Customers can reach support via email (support@shopmax.in), phone (1800-123-4567), '
        'or live chat on the website/app. '
        'Typical response time is within 24 hours for email queries and immediate for calls/chat. '
        'Support assists with orders, returns, refunds, warranty claims, and technical issues.'
    )
}
]


print(f'{len(DOCUMENTS)} documents loaded into knowledge base.')
for d in DOCUMENTS:
    print(f"  [{d['id']}] {d['topic']} — {len(d['text'])} chars")

20 documents loaded into knowledge base.
  [doc_001] Wireless Headphones — 625 chars
  [doc_002] Smart Watch — 595 chars
  [doc_003] Mechanical Keyboard — 595 chars
  [doc_004] Portable Bluetooth Speaker — 501 chars
  [doc_005] USB-C Fast Charger — 525 chars
  [doc_006] Laptop Stand — 560 chars
  [doc_013] Power Bank — 504 chars
  [doc_014] True Wireless Earbuds — 490 chars
  [doc_015] Gaming Mouse — 448 chars
  [doc_016] External SSD — 427 chars
  [doc_007] Return Policy — 674 chars
  [doc_008] Shipping Policy — 606 chars
  [doc_009] Refund Policy — 640 chars
  [doc_010] Warranty Policy — 772 chars
  [doc_011] Payment Methods — 609 chars
  [doc_012] Order Tracking and Cancellation — 723 chars
  [doc_017] Exchange Policy — 450 chars
  [doc_018] Account and Security Policy — 407 chars
  [doc_019] Privacy Policy — 396 chars
  [doc_020] Customer Support Policy — 364 chars


## Part 1 — Build VectorDB & Verify Retrieval

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

def load_vectordb():
    embedding = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
    texts  = [d['text']  for d in DOCUMENTS]
    metas  = [{'topic': d['topic'], 'id': d['id']} for d in DOCUMENTS]
    db = Chroma.from_texts(texts=texts, embedding=embedding, metadatas=metas)
    return db

vectordb = load_vectordb()
print('VectorDB built successfully.')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VectorDB built successfully.


In [5]:
def simple_rerank(query, docs):
    query = query.lower()

    scored = []
    for doc in docs:
        score = 0
        text = doc.page_content.lower()
        topic = doc.metadata.get('topic', '').lower()

        for word in query.split():
            if word in text:
                score += 1

        if topic in query:
            score += 3

        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)

    return [doc for _, doc in scored[:1]]

test_queries = [
    'What is the return policy?',
    'How long does shipping take?',
    'Tell me about the wireless headphones',
    'How do I cancel my order?',
    'What payment methods are accepted?',
]

print('Retrieval Verification\n')

for q in test_queries:
    docs = vectordb.similarity_search(q, k=5)
    docs = simple_rerank(q, docs)

    topics = [d.metadata['topic'] for d in docs]

    print(f'Q: {q}')
    print(f'   Retrieved: {topics}\n')

print('Retrieval verified. Proceeding to graph construction.')

Retrieval Verification

Q: What is the return policy?
   Retrieved: ['Return Policy']

Q: How long does shipping take?
   Retrieved: ['Shipping Policy']

Q: Tell me about the wireless headphones
   Retrieved: ['Wireless Headphones']

Q: How do I cancel my order?
   Retrieved: ['Order Tracking and Cancellation']

Q: What payment methods are accepted?
   Retrieved: ['Payment Methods']

Retrieval verified. Proceeding to graph construction.


## Part 2 — State Design

In [6]:
from typing import TypedDict, List, Optional

class FinState(TypedDict):

    question:     str
    messages:     List[dict]
    route:        str
    retrieved:    str
    sources:      List[str]
    tool_result:  str
    answer:       str
    faithfulness: float
    eval_retries: int
    user_name:    Optional[str]

print('FinState defined.')

FinState defined.


## Part 3 — Node Functions

In [7]:
from langchain_groq import ChatGroq

def get_llm(temperature: float = 0.1):
    return ChatGroq(
        model='llama-3.1-8b-instant',
        api_key=os.environ['GROQ_API_KEY'],
        temperature=temperature,
    )

print('LLM helper defined.')

LLM helper defined.


In [8]:
def memory_node(state: FinState) -> dict:
    msgs = state.get('messages', [])
    msgs = msgs + [{'role': 'user', 'content': state['question']}]
    msgs = msgs[-6:]

    user_name = state.get('user_name')
    match = re.search(r'my name is ([A-Za-z]+)', state['question'], re.IGNORECASE)
    if match:
        user_name = match.group(1).capitalize()

    return {
        **state,
        'messages':  msgs,
        'user_name': user_name,
    }

_s = {'question': 'My name is Priya. What is the return policy?',
      'messages': [], 'route': '', 'retrieved': '', 'sources': [],
      'tool_result': '', 'answer': '', 'faithfulness': 1.0,
      'eval_retries': 0, 'user_name': None}
_out = memory_node(_s)
print('memory_node OK:', _out['user_name'], '|', len(_out['messages']), 'msgs')

memory_node OK: Priya | 1 msgs


In [9]:
def router_node(state: FinState) -> dict:
    llm = get_llm()
    prompt = f"""You are a routing agent for an e-commerce customer support bot.

Classify the user question into EXACTLY ONE of these routes:
- retrieve  : question is about products, returns, shipping, refunds, warranty, payment, order tracking, or cancellation
- tool      : question asks for today's date, current time, or asks to calculate a discount
- skip      : greeting, small talk, or anything not related to shopping or products

Reply with ONE word only — no punctuation, no explanation.

Question: {state['question']}
Route:"""
    raw = llm.invoke(prompt).content.strip().lower().split()[0]
    route = raw if raw in ('retrieve', 'tool', 'skip') else 'retrieve'
    return {**state, 'route': route}

_out = router_node({**_s, 'question': 'How do I return my headphones?'})
print('router_node OK:', _out['route'])

router_node OK: retrieve


In [10]:
import re

def retrieval_node(state: FinState) -> dict:
    docs = vectordb.similarity_search(state['question'], k=3)
    context_parts = []
    sources = []
    for d in docs:
        topic = d.metadata.get('topic', 'Info')
        context_parts.append(f'[{topic}]\n{d.page_content}')
        sources.append(topic)
    retrieved = '\n\n'.join(context_parts)
    return {**state, 'retrieved': retrieved, 'sources': sources}

_out = retrieval_node({**_s, 'question': 'What is the shipping cost?'})
print('retrieval_node OK:', _out['sources'])

retrieval_node OK: ['Shipping Policy', 'Exchange Policy', 'USB-C Fast Charger']


In [11]:
def skip_retrieval_node(state: FinState) -> dict:
    return {**state, 'retrieved': '', 'sources': []}

print('skip_retrieval_node OK')

skip_retrieval_node OK


In [25]:
import re

def discount_calculator(question: str) -> str:
    q = question.lower()

    m = re.search(r'price\s*(?:is)?\s*(\d+)\s*after\s*(\d+)\s*%', q)
    if m:
        final_price = float(m.group(1))
        pct = float(m.group(2))
        original = final_price / (1 - pct/100)
        return (
            f"If Rs {final_price:.2f} is after a {pct:.0f}% discount, "
            f"the original price was Rs {original:,.2f}."
        )
    m = re.search(r'(\d+)\s*%\s*(?:on|of)\s*rs?\s*(\d+)', q)
    if m:
        pct = float(m.group(1))
        price = float(m.group(2))
        disc = price * pct / 100
        return (
            f"A {pct:.0f}% discount on Rs {price:,.2f} saves Rs {disc:,.2f}. "
            f"Final price: Rs {price-disc:,.2f}."
        )

    return "Please specify like '10% on Rs 1000' or 'price is 900 after 10% discount'."

def tool_node(state: FinState) -> dict:
    question = state['question']
    tool_result = ''
    sources = []

    if re.search(r'date|time|today', question, re.IGNORECASE):
        try:
            now = datetime.datetime.now()
            tool_result = f"Today is {now.strftime('%A, %d %B %Y')}. Current time is {now.strftime('%I:%M %p')} (IST)."
            sources = ['datetime']
        except Exception as e:
            tool_result = f'Could not retrieve date/time: {str(e)}'

    elif re.search(r'discount|%|percent|off', question, re.IGNORECASE):
        tool_result = discount_calculator(question)
        sources = ['discount_calculator']

    else:
        tool_result = "I'm not sure which tool to use for that request."

    return {**state, 'tool_result': tool_result, 'retrieved': '', 'sources': sources}

_out = tool_node(_s)
print('tool_node OK:', _out['tool_result'])

tool_node OK: I'm not sure which tool to use for that request.


In [13]:
def answer_node(state: FinState) -> dict:
    llm = get_llm(temperature=0.1)

    retrieved   = state.get('retrieved', '').strip()
    tool_result = state.get('tool_result', '').strip()
    user_name   = state.get('user_name', '')
    retries     = state.get('eval_retries', 0)

    name_part = f"The customer's name is {user_name}. Address them naturally.\n" if user_name else ""

    retry_instruction = ""
    if retries > 0:
        retry_instruction = (
            f"\nIMPORTANT: This is retry attempt {retries}. "
            "Your previous answer was incorrect or not grounded in the provided information. "
            "You MUST strictly follow all rules and not hallucinate.\n"
        )

    context_blocks = []
    if retrieved:
        context_blocks.append(f"[KNOWLEDGE BASE]\n{retrieved}")
    if tool_result:
        context_blocks.append(f"[TOOL RESULT]\n{tool_result}")

    context_section = "\n\n".join(context_blocks)

    prompt = f"""You are an intelligent customer support assistant for ShopMax (an e-commerce platform).

{name_part}
{retry_instruction}

SYSTEM ROLE:
You must correctly handle THREE types of user requests:

1. CHITCHAT
   - Greetings, small talk, casual conversation
   - Respond naturally, warmly, and briefly

2. TOOL-BASED
   - Calculations (discounts), date/time, utility queries
   - If a TOOL RESULT is provided → you MUST use it directly
   - Do NOT recompute or modify tool outputs

3. KNOWLEDGE-BASED (RAG)
   - Questions about ShopMax PRODUCTS or POLICIES


KNOWLEDGE STRUCTURE:

PRODUCTS:
- Product details such as features, specifications, price, discount, compatibility, warranty

POLICIES:
- Rules such as return policy, refund timelines, shipping details, payment methods, warranty terms


STRICT GROUNDING RULES:

When answering from KNOWLEDGE BASE:

1. Use ONLY the provided context
2. Do NOT use prior knowledge
3. Do NOT guess, infer, or assume missing details
4. Do NOT combine unrelated pieces of information
5. Extract exact facts (numbers, prices, durations, limits) precisely as written
6. If the answer is NOT explicitly present, respond EXACTLY:
   "I don't have that information in my knowledge base."


TOOL PRIORITY RULE:

If TOOL RESULT is present:
- Use it directly as the answer
- Do NOT add explanations, reasoning, or extra text
- Output only the final result in a natural sentence
- Do NOT override it with knowledge base information


ANSWER STYLE RULES:

- Be concise, clear, and helpful
- Do NOT repeat the question
- Do NOT mention "context", "documents", or internal reasoning
- Do NOT use emojis
- Address the user naturally if their name is available
-Do NOT add introductory phrases like:
"I have the information you need"
"Based on the information"
"Here is the answer"

Start directly with the answer.


DECISION LOGIC:

Follow this EXACT order:

Step 1: If TOOL RESULT exists → use it
Step 2: Else if question is about PRODUCTS or POLICIES → use KNOWLEDGE BASE
Step 3: Else → treat as CHITCHAT


CONTEXT:
{context_section if context_section else "No additional context available."}


CUSTOMER QUESTION:

{state['question']}


FINAL ANSWER:
"""

    answer = llm.invoke(prompt).content.strip()

    return {**state, 'answer': answer}

In [14]:
def eval_node(state: FinState) -> dict:
    retrieved = state.get('retrieved', '')
    if not retrieved:
        return {**state, 'faithfulness': 1.0}

    llm = get_llm()
    prompt = f"""Rate how faithful the assistant answer is to the given context.
Score 1.0 = answer uses only context facts. Score 0.0 = answer invents information not in context.

Context:
{retrieved}

Answer:
{state['answer']}

Reply with a single decimal number between 0.0 and 1.0. Nothing else."""

    raw = llm.invoke(prompt).content.strip()
    try:
        score = float(re.findall(r'\d+\.?\d*', raw)[0])
        score = min(max(score, 0.0), 1.0)
    except Exception:
        score = 0.5

    retries = state.get('eval_retries', 0) + 1
    return {**state, 'faithfulness': score, 'eval_retries': retries}

print('eval_node defined.')

eval_node defined.


In [15]:
def save_node(state: FinState) -> dict:
    msgs = state.get('messages', [])
    msgs = msgs + [{'role': 'assistant', 'content': state['answer']}]
    msgs = msgs[-6:]
    return {**state, 'messages': msgs}

print('save_node defined.')

save_node defined.


## Part 4 — Graph Assembly

In [16]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

def route_decision(state: FinState) -> str:
    r = state.get('route', 'retrieve')
    if r == 'tool':     return 'tool'
    if r == 'skip':     return 'skip'
    return 'retrieve'

def eval_decision(state: FinState) -> str:
    score   = state.get('faithfulness', 1.0)
    retries = state.get('eval_retries', 0)
    if score < 0.7 and retries < 2:
        return 'answer'   # retry
    return 'save'

def build_graph():
    memory = MemorySaver()
    g = StateGraph(FinState)

    g.add_node('memory',    memory_node)
    g.add_node('router',    router_node)
    g.add_node('retrieval', retrieval_node)
    g.add_node('skip',      skip_retrieval_node)
    g.add_node('tool',      tool_node)
    g.add_node('answer',    answer_node)
    g.add_node('eval',      eval_node)
    g.add_node('save',      save_node)

    g.set_entry_point('memory')
    g.add_edge('memory', 'router')

    g.add_conditional_edges('router', route_decision, {
        'retrieve': 'retrieval',
        'skip':     'skip',
        'tool':     'tool',
    })

    g.add_edge('retrieval', 'answer')
    g.add_edge('skip',      'answer')
    g.add_edge('tool',      'answer')
    g.add_edge('answer',    'eval')

    g.add_conditional_edges('eval', eval_decision, {
        'answer': 'answer',
        'save':   'save',
    })

    g.add_edge('save', END)
    return g.compile(checkpointer=memory)

app = build_graph()
print('Graph compiled successfully.')

Graph compiled successfully.


## Part 5 — Testing

In [17]:
def ask(question: str, thread_id: str = 'test') -> dict:
    state = {
        'question':     question,
        'messages':     [],
        'route':        '',
        'retrieved':    '',
        'sources':      [],
        'tool_result':  '',
        'answer':       '',
        'faithfulness': 1.0,
        'eval_retries': 0,
        'user_name':    None,
    }
    config = {'configurable': {'thread_id': thread_id}}
    result = app.invoke(state, config)
    return result

print('ask() helper defined.')

ask() helper defined.


In [18]:
TEST_QUESTIONS = [
    ('What is the return policy?',                                    'domain: policy'),
    ('How much does express shipping cost?',                          'domain: shipping'),
    ('Tell me the features of the wireless headphones.',              'domain: product'),
    ('Is the Smart Watch suitable for swimming?',                     'domain: product'),
    ('What is the price of the 65W USB-C charger?',                   'domain: product'),
    ('What is the warranty period for the Mechanical Keyboard?',      'domain: policy'),
    ('What is the current date and time?',                            'tool: datetime'),
    ('Can you calculate a 20% discount on Rs 2500?',                  'tool: discount'),
    ('Hello, how are you?',                                           'skip: greeting'),
    ('What is the capital of France?',                                'skip: out-of-scope'),
    ('I heard ShopMax has free international shipping.',              'red-team: false premise'),
    ('What payment methods can I use?',                               'domain: policy'),
    ('How can I cancel an order?',                                    'domain: policy'),
]

print('Test Run\n')
results = []
for question, category in TEST_QUESTIONS:
    r = ask(question, thread_id='test_run')
    score = r.get('faithfulness', 1.0)
    route = r.get('route', '?')
    answer = r.get('answer', '')
    status = 'PASS' if score >= 0.7 or not r.get('retrieved') else 'CHECK'
    results.append((question, category, route, score, status, answer))
    print(f'[{status}] [{category}] Route={route} | Faithfulness={score:.2f}')
    print(f'  Q: {question}')
    print(f'  A: {answer[:120]}...\n')


Test Run

[PASS] [domain: policy] Route=retrieve | Faithfulness=1.00
  Q: What is the return policy?
  A: Customers may return any product within 7 days of delivery for a full refund, provided the item is unused, in its origin...

[PASS] [domain: shipping] Route=retrieve | Faithfulness=1.00
  Q: How much does express shipping cost?
  A: Express shipping costs Rs 99 regardless of order value....

[PASS] [domain: product] Route=retrieve | Faithfulness=1.00
  Q: Tell me the features of the wireless headphones.
  A: The ShopMax Wireless Headphones (Model WH-300) use Bluetooth 5.3 for a stable, low-latency connection up to 10 metres. T...

[PASS] [domain: product] Route=retrieve | Faithfulness=1.00
  Q: Is the Smart Watch suitable for swimming?
  A: The ShopMax Smart Watch (Model SW-200) is IP68 waterproof and safe for swimming up to 50 metres....

[PASS] [domain: product] Route=retrieve | Faithfulness=1.00
  Q: What is the price of the 65W USB-C charger?
  A: The price of the 65W USB-C cha

In [19]:
MEMORY_THREAD = 'memory_test_' + str(uuid.uuid4())

print('=== Memory Test ===\n')
q1 = ask('My name is Rahul.', thread_id=MEMORY_THREAD)
print('Q1:', q1['question'])
print('A1:', q1['answer'])
print()

q2 = ask('What is the return policy?', thread_id=MEMORY_THREAD)
print('Q2:', q2['question'])
print('A2:', q2['answer'])
print()

q3 = ask('Can you remind me what my name is?', thread_id=MEMORY_THREAD)
print('Q3:', q3['question'])
print('A3 (should mention Rahul):', q3['answer'])
print()
print('Memory test complete. Check that A3 references "Rahul".')

=== Memory Test ===

Q1: My name is Rahul.
A1: Hi Rahul, how's your day going so far?

Q2: What is the return policy?
A2: Customers may return any product within 7 days of delivery for a full refund, provided the item is unused, in its original packaging, and all accessories and manuals are included.

Q3: Can you remind me what my name is?
A3 (should mention Rahul): I'm happy to help you. Unfortunately, I don't have any information about your name in our system.

Memory test complete. Check that A3 references "Rahul".


## Part 6 — RAGAS Baseline Evaluation

In [20]:
EVAL_PAIRS = [
    {
        'question':     'How many days do I have to return a product?',
        'ground_truth': 'Products can be returned within 7 days of delivery if unused and in original packaging.',
    },
    {
        'question':     'How much does express delivery cost?',
        'ground_truth': 'Express delivery costs Rs 99 regardless of order value and takes 1-2 business days.',
    },
    {
        'question':     'What is the battery life of the Smart Watch?',
        'ground_truth': 'The Smart Watch has 7 days of battery life in normal mode and 3 days with always-on display.',
    },
    {
        'question':     'What payment methods are accepted?',
        'ground_truth': 'ShopMax accepts credit cards, debit cards, UPI, net banking, Cash on Delivery, and ShopMax Wallet. EMI is available on orders above Rs 3,000.',
    },
    {
        'question':     'How do I cancel my order?',
        'ground_truth': 'Orders can be cancelled for free within 12 hours of placement if not yet dispatched by going to My Orders and clicking Cancel Order.',
    },
]

eval_data = {'question': [], 'answer': [], 'contexts': [], 'ground_truth': []}

print('Running RAGAS evaluation...\n')
for pair in EVAL_PAIRS:
    r = ask(pair['question'], thread_id='ragas_eval')
    eval_data['question'].append(pair['question'])
    eval_data['answer'].append(r.get('answer', ''))
    eval_data['contexts'].append([r.get('retrieved', '')])
    eval_data['ground_truth'].append(pair['ground_truth'])
    print(f"Q: {pair['question']}")
    print(f"A: {r.get('answer', '')[:100]}...\n")

try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    dataset = Dataset.from_dict(eval_data)
    ragas_result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision])
    print('\n=== RAGAS Baseline Scores ===')
    print(ragas_result)
except Exception as e:
    # Manual fallback
    print(f'RAGAS auto-eval failed ({e}). Using manual LLM faithfulness fallback.\n')
    llm = get_llm()
    scores = []
    for i, pair in enumerate(EVAL_PAIRS):
        ctx  = eval_data['contexts'][i][0]
        ans  = eval_data['answer'][i]
        gt   = pair['ground_truth']
        prompt = f"""Rate faithfulness of the answer to the context. Reply with a number 0.0-1.0 only.
Context: {ctx[:500]}
Answer: {ans}
Score:"""
        raw = llm.invoke(prompt).content.strip()
        try:
            s = float(re.findall(r'\d+\.?\d*', raw)[0])
        except:
            s = 0.5
        scores.append(s)
        print(f"  [{i+1}] Faithfulness: {s:.2f} | Q: {pair['question']}")
    print(f'\nMean manual faithfulness: {sum(scores)/len(scores):.2f}')

Running RAGAS evaluation...

Q: How many days do I have to return a product?
A: You have 7 days to return a product....

Q: How much does express delivery cost?
A: Express delivery costs Rs 99....

Q: What is the battery life of the Smart Watch?
A: The battery life of the Smart Watch is 7 days in normal mode, 3 days with always-on display enabled....

Q: What payment methods are accepted?
A: ShopMax accepts the following payment methods: Credit cards (Visa, Mastercard, Amex, RuPay), debit c...

Q: How do I cancel my order?
A: You can cancel your order for free within 12 hours of placement if it hasn't been dispatched yet. To...



/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_3073/2103907213.py:41: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_3073/2103907213.py:41: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections i

RAGAS auto-eval failed (The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable). Using manual LLM faithfulness fallback.

  [1] Faithfulness: 0.90 | Q: How many days do I have to return a product?
  [2] Faithfulness: 0.90 | Q: How much does express delivery cost?
  [3] Faithfulness: 0.90 | Q: What is the battery life of the Smart Watch?
  [4] Faithfulness: 0.90 | Q: What payment methods are accepted?
  [5] Faithfulness: 0.90 | Q: How do I cancel my order?

Mean manual faithfulness: 0.90


## Part 7 — Streamlit UI (write app.py)

In [39]:
APP_CODE = r'''
import os, re, uuid, datetime
from typing import TypedDict, List, Optional
import streamlit as st
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

#PAGE CONFIG
st.set_page_config(
    page_title="ShopMax Support",
    page_icon="S",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.sidebar.empty()

st.markdown(
    """
    <style>
    [data-testid="stSidebar"][aria-expanded="false"] {
        display: block !important;
        width: 300px !important;
    }
    </style>
    """,
    unsafe_allow_html=True
)
#CSS + JS
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:opsz,wght@9..40,300;9..40,400;9..40,500;9..40,600&display=swap');

#Design Tokens
:root {
    --bg-base:        #18230F;
    --bg-sidebar:     #2F3E33;
    --bg-sidebar-hov: #2A3630;
    --bg-input:       #1A2812;

    --user-bubble:    #2C3E5A;
    --user-bdr:       #3A5070;
    --bot-bubble:     #D58A94;
    --bot-bdr:        #E1AD9D;

    --accent:         #91A6FF;
    --accent-glow:    rgba(145,166,255,0.22);
    --assistant:      #D58A94;

    --text-primary:   #FFFFFF;
    --text-muted:     #7A9070;
    --sb-text:        #FEFAE0;
    --sb-muted:       #B8C4A8;

    --success:        #6BCB77;
    --border:         rgba(145,166,255,0.10);
    --border-strong:  rgba(145,166,255,0.20);
    --shadow:         0 4px 16px rgba(0,0,0,0.40);

    --r-sm: 6px;
    --r-lg: 18px;
    --ease: 160ms ease;
}

#Global
body {
    background-color: var(--bg-base);
}
.block-container {
    padding-top: 0.5rem !important;
    padding-bottom: 0rem !important;
    max-width: 100% !important;
    font-family: 'DM Sans', sans-serif !important;
    color: var(--text-primary) !important;
}

#Scrollbar
::-webkit-scrollbar { width:5px; height:5px; }
::-webkit-scrollbar-track { background:var(--bg-base); }
::-webkit-scrollbar-thumb { background:var(--border-strong); border-radius:99px; }
::-webkit-scrollbar-thumb:hover { background:var(--accent); }

#Hide Streamlit chrome
#MainMenu, footer { visibility:hidden !important; }
header { visibility:hidden !important; }
[data-testid="stToolbar"]  { visibility: visible !important; }
[data-testid="stDecoration"]{ display:none !important; }

#SIDEBAR
[data-testid="stSidebar"] {
    background-color: var(--bg-sidebar) !important;
    border-right: 1px solid rgba(145,166,255,0.08) !important;
    box-shadow: 4px 0 24px rgba(0,0,0,0.30) !important;
    width: 300px !important;
    flex-shrink: 0 !important;
}
[data-testid="stSidebar"] > div:first-child {
    padding: 1.25rem 0.85rem 2rem !important;
}
section[data-testid="stSidebar"],
section[data-testid="stSidebar"] * {
    color: var(--sb-text) !important;
}

#Brand
.sb-brand {
    display:flex; align-items:center; gap:10px; margin-bottom:1.25rem;
}
.sb-logo {
    width:38px; height:38px; flex-shrink:0;
    background: linear-gradient(135deg,#91A6FF,#6B8EFF);
    border-radius:10px;
    display:flex; align-items:center; justify-content:center;
    box-shadow:0 4px 12px rgba(145,166,255,0.30);
    font-size:22px; line-height:1;
}
.sb-title {
    font-family:'DM Serif Display',serif !important;
    font-size:1.3rem; letter-spacing:-0.02em; line-height:1.1;
}
.sb-subtitle {
    font-size:0.65rem; letter-spacing:0.07em;
    text-transform:uppercase; color:var(--sb-muted) !important;
    margin-top:2px;
}

#Section labels
.sb-label {
    display:block;
    font-size:0.60rem !important; font-weight:700 !important;
    letter-spacing:0.14em !important; text-transform:uppercase !important;
    color:var(--text-muted) !important;
    margin: 1.1rem 0 0.45rem 2px !important;
}

#Search
section[data-testid="stSidebar"] [data-testid="stTextInput"] input {
    background-color: rgba(0,0,0,0.28) !important;
    border: 1px solid var(--border-strong) !important;
    border-radius: var(--r-sm) !important;
    color: var(--sb-text) !important;
    font-family:'DM Sans',sans-serif !important;
    font-size:0.82rem !important;
}
section[data-testid="stSidebar"] [data-testid="stTextInput"] input:focus {
    border-color:var(--accent) !important;
    box-shadow:0 0 0 3px var(--accent-glow) !important;
    outline:none !important;
}
section[data-testid="stSidebar"] [data-testid="stTextInput"] input::placeholder {
    color:var(--text-muted) !important;
}

#All sidebar buttons
button[data-testid="collapsedControl"] {
    display: flex !important;
    visibility: visible !important;
    opacity: 1 !important;
    z-index: 9999 !important;
}
section[data-testid="stSidebar"] .stButton > button {
    background-color: transparent !important;
    color: var(--sb-muted) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--r-sm) !important;
    padding: 0.38rem 0.75rem !important;
    font-size: 0.79rem !important;
    font-family:'DM Sans',sans-serif !important;
    width: 100% !important;
    text-align: left !important;
    margin-bottom: 0.18rem !important;
    transition: all var(--ease) !important;
}
section[data-testid="stSidebar"] .stButton > button:hover {
    background-color: var(--bg-sidebar-hov) !important;
    border-color: var(--accent) !important;
    color: var(--sb-text) !important;
    transform: translateX(3px) !important;
    box-shadow: 0 2px 8px rgba(0,0,0,0.20) !important;
}

#Clear button
section[data-testid="stSidebar"] .stButton:last-of-type > button {
    background-color: rgba(213,138,148,0.07) !important;
    border-color: rgba(213,138,148,0.30) !important;
    color: var(--assistant) !important;
}
section[data-testid="stSidebar"] .stButton:last-of-type > button:hover {
    background-color: rgba(213,138,148,0.16) !important;
    border-color: var(--assistant) !important;
    color: #F0A8B0 !important;
    transform: none !important;
}

#Expanders

section[data-testid="stSidebar"] [data-testid="stExpander"] {
    background-color: rgba(0,0,0,0.15) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--r-sm) !important;
    margin-bottom:0.32rem !important;
    overflow:hidden !important;
}
section[data-testid="stSidebar"] [data-testid="stExpander"] summary {
    color: var(--sb-muted) !important;
    font-size:0.8rem !important; font-weight:500 !important;
    padding:0.48rem 0.75rem !important;
}
section[data-testid="stSidebar"] [data-testid="stExpander"] summary:hover {
    color:var(--sb-text) !important;
}
section[data-testid="stSidebar"] hr {
    border-color:rgba(145,166,255,0.08) !important;
    margin:0.75rem 0 !important;
}

#MAIN HEADER
.main-header {
    display:flex; align-items:center; justify-content:space-between;
    padding-bottom:0.85rem;
    border-bottom:1px solid var(--border);
    margin-bottom:0.6rem;
}
.main-title {
    font-family:'DM Serif Display',serif;
    font-size:1.9rem; letter-spacing:-0.04em; line-height:1;
    background:linear-gradient(130deg,#FFFFFF 0%,var(--accent) 100%);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent;
    background-clip:text;
}
.main-subtitle {
    color:var(--text-muted); font-size:0.80rem;
    margin-top:0.3rem;
}
.status-pill {
    display:flex; align-items:center; gap:6px;
    background:rgba(107,203,119,0.10);
    border:1px solid rgba(107,203,119,0.28);
    border-radius:99px; padding:4px 12px;
    font-size:0.70rem; font-weight:700;
    color:var(--success); letter-spacing:0.07em;
    white-space:nowrap;
}
.status-dot {
    width:6px; height:6px;
    background:var(--success); border-radius:50%;
    animation:pdot 2s ease-in-out infinite;
}
@keyframes pdot {
    0%,100%{ opacity:1; transform:scale(1); }
    50%    { opacity:0.35; transform:scale(0.75); }
}

#Info chips
.info-strip { display:flex; gap:8px; flex-wrap:wrap; margin-bottom:1rem; }
.info-chip {
    display:flex; align-items:center; gap:4px;
    background:rgba(145,166,255,0.06);
    border:1px solid var(--border-strong);
    border-radius:99px; padding:3px 10px;
    font-size:0.69rem; color:var(--text-muted);
}
.info-chip b { color:var(--accent); font-weight:500; }

#AVATAR

#Avatar background colours
[data-testid="stChatMessage"] [data-testid="chatAvatarIcon-user"] {
    background: linear-gradient(135deg,#5B78CC,#91A6FF) !important;
    box-shadow:0 2px 8px rgba(145,166,255,0.30) !important;
}
[data-testid="stChatMessage"] [data-testid="chatAvatarIcon-assistant"] {
    background: linear-gradient(135deg,#8A4550,#D58A94) !important;
    box-shadow:0 2px 8px rgba(213,138,148,0.30) !important;
}

#CHAT MESSAGES — BUBBLES

#Chat row layout
.chat-row {
    display: flex;
    width: 100%;
    margin-bottom: 0.85rem;
}

#Alignment
.chat-row.user {
    justify-content: flex-end;
}
.chat-row.bot {
    justify-content: flex-start;
}

#Bubble base
.chat-bubble {
    background: #1E2E18 !important;
    max-width: 70%;
    padding: 0.78rem 1.1rem;
    border-radius: 15px;
    font-size: 0.88rem;
    line-height: 1.6;
    box-shadow: 0 4px 16px rgba(0,0,0,0.40);
}

#User bubble
.chat-row.user .chat-bubble {
    background: linear-gradient(135deg, #2C3E5A, #243350) !important;
    border: 1px solid #3A5070;
    border-radius: 15px 15px 0 15px;
}

#Bot bubble
.chat-row.bot .chat-bubble {
    background: linear-gradient(135deg, #D58A94, #E1AD9D) !important;
    border: 1px solid #2E4228;
    border-radius: 15px 15px 15px 0;
}

#Typing indicator
.typing-wrap {
    display:flex; align-items:center; gap:5px;
    padding:0.55rem 0.9rem;
    background: linear-gradient(135deg, #1E2E18, #182415);
    border:1px solid #2E4228;
    border-radius:15px 15px 15px 0;
    width:fit-content; margin-bottom:0.75rem;
    box-shadow:var(--shadow);
}
.tdot {
    width:7px; height:7px;
    background:var(--assistant); border-radius:50%;
    animation:tdbounce 1.25s ease-in-out infinite;
}
.tdot:nth-child(2){ animation-delay:.22s; }
.tdot:nth-child(3){ animation-delay:.44s; }
@keyframes tdbounce {
    0%,80%,100%{ transform:translateY(0); opacity:0.4; }
    40%        { transform:translateY(-6px); opacity:1; }
}

#CHAT INPUT
[data-testid="stChatInput"] {
    border-radius: 12px !important;
}
[data-testid="stChatInput"] textarea {
    background-color: transparent !important;
    color: var(--text-primary) !important;
    font-family:'DM Sans',sans-serif !important;
    font-size:0.9rem !important;
    caret-color:var(--accent) !important;
}
[data-testid="stChatInput"] textarea::placeholder {
    color:var(--text-muted) !important; font-style:italic;
}
[data-testid="stChatInput"] button {
    background: linear-gradient(135deg,var(--accent),#6B8EFF) !important;
    border-radius: 10px !important;
    transition: transform var(--ease), box-shadow var(--ease) !important;
}
[data-testid="stChatInput"] button:hover {
    transform:scale(1.08) !important;
    box-shadow:0 4px 14px var(--accent-glow) !important;
}

/* Prevent layout shift when sidebar loads */
section.main > div {
    padding-left: 0rem !important;
}
</style>
""", unsafe_allow_html=True)


#STATE
class FinState(TypedDict):
    question:     str
    messages:     List[dict]
    route:        str
    retrieved:    str
    sources:      List[str]
    tool_result:  str
    answer:       str
    faithfulness: float
    eval_retries: int
    user_name:    Optional[str]


#KNOWLEDGE BASE
DOCUMENTS = [
    {
        'id':'doc_001','category':'Electronics > Audio','topic':'Wireless Headphones',
        'sku':'WH-300','price':1999,'discount':'10%','stock_status':'In Stock',
        'text':(
            'The ShopMax Wireless Headphones (Model WH-300) use Bluetooth 5.3 for a stable, '
            'low-latency connection up to 10 metres. Battery life is 20 hours on a single charge, '
            'with a 2-hour fast-charge via USB-C. The headphones include active noise cancellation (ANC), '
            'a built-in microphone for calls, and 40mm drivers for rich bass. Weight is 250g. '
            'They are foldable and come with a hard carrying case. Available colours: Black, White, Navy Blue. '
            'Price: Rs 1,999. Currently available at a 10% discount. Covered by a 1-year manufacturer warranty '
            'against hardware defects. Compatible with iOS, Android, Windows, and macOS. Not waterproof.'
        )
    },
    {
        'id':'doc_002','category':'Electronics > Wearables','topic':'Smart Watch',
        'sku':'SW-200','price':2999,'discount':'10%','stock_status':'In Stock',
        'text':(
            'The ShopMax Smart Watch (Model SW-200) tracks steps, calories, sleep, heart rate, '
            'and blood oxygen (SpO2). It has a 1.4-inch AMOLED display with always-on mode. '
            'Battery life is 7 days in normal mode, 3 days with always-on display enabled. '
            'It is IP68 waterproof — safe for swimming up to 50 metres. Compatible with iOS 12+ and Android 8+. '
            'Includes 20+ workout modes. Built-in GPS for outdoor runs. '
            'Supports notifications for calls, SMS, and apps. Strap is interchangeable (22mm standard). '
            'Price: Rs 2,999. Currently available at a 10% discount. 1-year warranty.'
        )
    },
    {
        'id':'doc_003','category':'Electronics > Accessories','topic':'Mechanical Keyboard',
        'sku':'KB-75','price':3499,'discount':'10%','stock_status':'In Stock',
        'text':(
            'The ShopMax Mechanical Keyboard (Model KB-75) is a tenkeyless (TKL) wired keyboard with '
            'Cherry MX Red switches — linear, quiet, and ideal for both typing and gaming. '
            'It has per-key RGB backlighting with 15 preset lighting modes. Aluminium top plate and '
            'PBT double-shot keycaps. N-key rollover (NKRO). USB-A braided cable (1.8m). '
            'Layout: US ANSI. Compatible with Windows 10/11 and macOS. No driver required. '
            'Price: Rs 3,499. Currently available at a 10% discount. 2-year warranty.'
        )
    },
    {
        'id':'doc_004','category':'Electronics > Audio','topic':'Portable Bluetooth Speaker',
        'sku':'BS-50','price':2499,'discount':None,'stock_status':'In Stock',
        'text':(
            'The ShopMax Blast Speaker (Model BS-50) delivers 20W stereo sound with dual 10W drivers '
            'and a passive radiator for deep bass. Battery lasts 12 hours at 70% volume. '
            'IPX7 waterproof — fully submersible up to 1 metre for 30 minutes. '
            'Bluetooth 5.0 range up to 15 metres. Supports wired AUX input (3.5mm). '
            'Built-in microphone for speakerphone calls. Charges in 3 hours via USB-C. '
            'Dimensions: 18cm x 7cm. Weight: 600g. Colours: Red, Black, Forest Green. '
            'Price: Rs 2,499. 1-year warranty.'
        )
    },
    {
        'id':'doc_005','category':'Electronics > Charging','topic':'USB-C Fast Charger',
        'sku':'FC-65','price':1299,'discount':'5%','stock_status':'In Stock',
        'text':(
            'The ShopMax 65W USB-C GaN Charger (Model FC-65) supports USB Power Delivery 3.0. '
            'Two ports: USB-C (65W max) and USB-A (18W max, Quick Charge 3.0). '
            'GaN technology makes it 40% smaller than traditional chargers. '
            'Charges an iPhone 15 to 50% in 30 minutes. MacBook Air in under 2 hours. '
            'Universal voltage (100-240V). Cable not included. Price: Rs 1,299. 1-year warranty. Colour: White.'
        )
    },
    {
        'id':'doc_006','category':'Accessories','topic':'Laptop Stand',
        'sku':'LS-20','price':899,'discount':None,'stock_status':'In Stock',
        'text':(
            'The ShopMax ErgoRise Laptop Stand (Model LS-20) is anodised aluminium, supports 10-16 inch '
            'laptops up to 10kg. Height adjustable across 6 levels (10-20cm), angle 15-45 degrees. '
            'Folds flat — packed size 28cm x 22cm x 1.5cm, weight 500g. Silicone pads grip desk. '
            'No USB hub. Compatible with all brands. Price: Rs 899. 1-year warranty. Silver or Space Grey.'
        )
    },
    {
        'id':'doc_013','category':'Electronics > Power','topic':'Power Bank',
        'sku':'PB-20000','price':1499,'discount':None,'stock_status':'In Stock',
        'text':(
            'The ShopMax Power Bank (Model PB-20000) has 20,000mAh capacity. '
            '18W fast charging via USB-C PD and QC 3.0. 2 USB-A and 1 USB-C port. '
            'Charges a smartphone 4-5 times. Fully recharges in 6-7 hours with 18W charger. '
            'Protections against overcharge, over-discharge, and short circuit. '
            'Weight: 450g. Price: Rs 1,499. 1-year warranty. Colour: Black.'
        )
    },
    {
        'id':'doc_014','category':'Electronics > Audio','topic':'True Wireless Earbuds',
        'sku':'TWS-100','price':1799,'discount':None,'stock_status':'In Stock',
        'text':(
            'The ShopMax True Wireless Earbuds (Model TWS-100) feature Bluetooth 5.2 with auto-pairing '
            'and touch controls. 5 hours per charge, 20 extra hours from the case. '
            'ENC for clearer calls. IPX5 water resistance. USB-C fast charging (10 min = 1 hr). '
            '3 sizes of ear tips included. Price: Rs 1,799. 1-year warranty.'
        )
    },
    {
        'id':'doc_015','category':'Electronics > Gaming','topic':'Gaming Mouse',
        'sku':'GM-80','price':1299,'discount':None,'stock_status':'In Stock',
        'text':(
            'The ShopMax Gaming Mouse (Model GM-80) has a 16,000 DPI optical sensor, '
            '6 programmable buttons, RGB lighting, and 1000Hz polling rate. '
            'Right-handed ergonomic design. USB-A braided cable (1.6m). Windows and macOS compatible. '
            'Software required for customization. Weight: 120g. Price: Rs 1,299. 1-year warranty.'
        )
    },
    {
        'id':'doc_016','category':'Electronics > Storage','topic':'External SSD',
        'sku':'SSD-1TB','price':6999,'discount':None,'stock_status':'In Stock',
        'text':(
            'The ShopMax External SSD (Model SSD-1TB) offers 1TB storage, read up to 1050 MB/s, '
            'write up to 1000 MB/s. USB 3.2 Gen 2 / USB-C. Aluminium, shock-resistant. '
            'Compatible with Windows, macOS, Android OTG, and gaming consoles. '
            'Includes USB-C to C and USB-C to A cables. Weight: 90g. Price: Rs 6,999. 3-year warranty.'
        )
    },
    {
        'id':'doc_007','category':'Policy','topic':'Return Policy',
        'text':(
            'Customers may return any product within 7 days of delivery for a full refund, '
            'provided the item is unused, in its original packaging, with all accessories. '
            'Initiate via ShopMax account > Orders > Return. Pickup arranged in 1-2 business days, free. '
            'Not eligible: misused items, missing parts, returns after 7 days, opened software, digital downloads. '
            'Refunds processed within 5-7 business days after receipt and verification.'
        )
    },
    {
        'id':'doc_008','category':'Policy','topic':'Shipping Policy',
        'text':(
            'ShopMax ships across India. Standard delivery 3-5 business days; free above Rs 499, else Rs 49. '
            'Express delivery (1-2 days) costs Rs 99. Same-day delivery (Rs 149) available in Bangalore, '
            'Mumbai, Delhi, Hyderabad, Chennai for orders before 11am. '
            'Dispatched Mon-Sat. Tracking link sent via SMS and email. International shipping unavailable.'
        )
    },
    {
        'id':'doc_009','category':'Policy','topic':'Refund Policy',
        'text':(
            'Refunds processed within 5-7 business days after warehouse verification. '
            'Credit/debit card refunds to original card. UPI/net banking to original account. '
            'COD orders refunded via bank transfer — provide account number and IFSC. '
            'ShopMax Wallet refunds within 24 hours. Email confirmation sent when initiated. '
            'Contact support with order ID if refund not received in 7 business days.'
        )
    },
    {
        'id':'doc_010','category':'Policy','topic':'Warranty Policy',
        'text':(
            'All ShopMax products include a manufacturer warranty. Most: 1 year; KB-75: 2 years. '
            'Claims: contact support@shopmax.in or 1800-123-4567 within the warranty period. '
            'Covers: DOA units, manufacturing defects, component failure under normal use. '
            'Does NOT cover: physical damage, water damage (non-waterproof products), '
            'incorrect voltage damage, unauthorised repair. Proof of purchase required. '
            'Approved claims: free repair or replacement.'
        )
    },
    {
        'id':'doc_011','category':'Policy','topic':'Payment Methods',
        'text':(
            'ShopMax accepts: Credit cards (Visa, Mastercard, Amex, RuPay), debit cards, '
            'UPI (GPay, PhonePe, Paytm, BHIM), net banking (50+ banks), '
            'Cash on Delivery (up to Rs 10,000), ShopMax Wallet. '
            'EMI on orders above Rs 3,000 via eligible credit cards (3/6/9/12 months). '
            'BNPL via ZestMoney and LazyPay. 256-bit SSL encryption. Prices inclusive of 18% GST.'
        )
    },
    {
        'id':'doc_012','category':'Policy','topic':'Order Tracking and Cancellation',
        'text':(
            'Confirmation email/SMS sent immediately after order. Tracking link from courier '
            '(Delhivery, Ekart, BlueDart) shared after dispatch. Track at shopmax.in/track or via app. '
            'Cancel free within 12 hours if not dispatched: My Orders > Cancel Order. '
            'Dispatched orders cannot be cancelled — request a return after delivery. '
            'Pre-dispatch cancellation refunds processed within 24 hours. '
            'Issues: support@shopmax.in or 1800-123-4567.'
        )
    },
    {
        'id':'doc_017','topic':'Exchange Policy',
        'text':(
            'ShopMax allows exchanges within 7 days of delivery for defective or damaged-on-arrival items. '
            'Request via account > Orders. Replacement shipped after pickup and inspection. '
            'Subject to stock availability; full refund issued if unavailable. '
            'Return with original packaging and all accessories.'
        )
    },
    {
        'id':'doc_018','topic':'Account and Security Policy',
        'text':(
            'Customers are responsible for keeping ShopMax account credentials confidential. '
            'ShopMax uses 256-bit SSL encryption. Do not share OTPs or passwords. '
            'Report suspicious activity to support@shopmax.in. '
            'ShopMax is not liable for unauthorised access due to user negligence.'
        )
    },
    {
        'id':'doc_019','topic':'Privacy Policy',
        'text':(
            'ShopMax collects name, contact details, and payment info to process orders and improve services. '
            'Data stored securely, not sold to third parties. '
            'Shared with logistics partners for delivery. Customers can request data deletion via support. '
            'Cookies used to enhance browsing and personalise recommendations.'
        )
    },
    {
        'id':'doc_020','topic':'Customer Support Policy',
        'text':(
            'ShopMax customer support: Monday-Saturday, 9am-7pm IST. '
            'Email: support@shopmax.in | Phone: 1800-123-4567 (toll-free) | Live chat on website/app. '
            'Response within 24 hours for email; immediate for calls and chat. '
            'Assists with orders, returns, refunds, warranty, and technical issues.'
        )
    },
]


#VECTOR DB]
@st.cache_resource(show_spinner="Loading knowledge base...")
def load_vectordb():
    emb   = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    texts = [d["text"] for d in DOCUMENTS]
    metas = [{"topic": d["topic"], "id": d["id"]} for d in DOCUMENTS]
    return Chroma.from_texts(texts=texts, embedding=emb, metadatas=metas)


#LLM
def get_llm(temperature: float = 0.1):
    return ChatGroq(
        model='llama-3.1-8b-instant',
        api_key=os.environ['GROQ_API_KEY'],
        temperature=temperature,
    )


#HELPERS
def simple_rerank(query, docs):
    q = query.lower()
    scored = []
    for doc in docs:
        score = sum(1 for w in q.split() if w in doc.page_content.lower())
        if doc.metadata.get('topic','').lower() in q:
            score += 3
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:3]]


def discount_calculator(question: str) -> str:
    q = question.lower()


    m = re.search(r'price\s*(?:is)?\s*(\d+)\s*after\s*(\d+)\s*%', q)
    if m:
        final_price = float(m.group(1))
        pct = float(m.group(2))
        original = final_price / (1 - pct/100)
        return (
            f"If Rs {final_price:.2f} is after a {pct:.0f}% discount, "
            f"the original price was Rs {original:,.2f}."
        )

    m = re.search(r'(\d+)\s*%\s*(?:on|of)\s*rs?\s*(\d+)', q)
    if m:
        pct = float(m.group(1))
        price = float(m.group(2))
        disc = price * pct / 100
        return (
            f"A {pct:.0f}% discount on Rs {price:,.2f} saves Rs {disc:,.2f}. "
            f"Final price: Rs {price-disc:,.2f}."
        )

    return "Please specify like '10% on Rs 1000' or 'price is 900 after 10% discount'."


#GRAPH NODES
def memory_node(state: FinState) -> dict:
    msgs = (state.get('messages') or []) + [{'role':'user','content':state['question']}]
    user_name = state.get('user_name')
    m = re.search(r'my name is ([A-Za-z]+)', state['question'], re.IGNORECASE)
    if m:
        user_name = m.group(1).capitalize()
    return {**state, 'messages': msgs[-6:], 'user_name': user_name}


def router_node(state: FinState) -> dict:
    llm = get_llm()
    prompt = (
        "You are a routing agent for an e-commerce support bot.\n"
        "Classify into ONE of: retrieve | tool | skip\n"
        "- retrieve: products, returns, shipping, refunds, warranty, payment, tracking, cancellation\n"
        "- tool: date, time, discount calculation\n"
        "- skip: greetings, small talk, off-topic\n"
        "Reply with ONE word only.\n\n"
        f"Question: {state['question']}\nRoute:"
    )
    raw = llm.invoke(prompt).content.strip().lower().split()[0]
    return {**state, 'route': raw if raw in ('retrieve','tool','skip') else 'retrieve'}


def retrieval_node(state: FinState) -> dict:
    db     = load_vectordb()
    docs   = db.similarity_search(state['question'], k=5)
    ranked = simple_rerank(state['question'], docs)
    parts, sources = [], []
    for d in ranked:
        topic = d.metadata.get('topic','Info')
        parts.append(f"[{topic}]\n{d.page_content}")
        sources.append(topic)
    return {**state, 'retrieved': '\n\n'.join(parts), 'sources': sources}


def skip_node(state: FinState) -> dict:
    return {**state, 'retrieved': '', 'sources': []}


def tool_node(state: FinState) -> dict:
    q = state['question']
    if re.search(r'date|time|today', q, re.IGNORECASE):
        now    = datetime.datetime.now()
        result = (f"Today is {now.strftime('%A, %d %B %Y')}. "
                  f"Current time is {now.strftime('%I:%M %p')} IST.")
        src    = ['datetime']
    elif re.search(r'discount|%|percent|off', q, re.IGNORECASE):
        result = discount_calculator(q)
        src    = ['discount_calculator']
    else:
        result = "I'm not sure which tool to use for that request."
        src    = []
    return {**state, 'tool_result': result, 'retrieved': '', 'sources': src}


def answer_node(state: FinState) -> dict:
    llm = get_llm(temperature=0.1)

    retrieved   = state.get('retrieved', '').strip()
    tool_result = state.get('tool_result', '').strip()
    user_name   = state.get('user_name', '')
    retries     = state.get('eval_retries', 0)

    name_part = f"The customer's name is {user_name}. Address them naturally.\n" if user_name else ""

    retry_instruction = ""
    if retries > 0:
        retry_instruction = (
            f"\nIMPORTANT: This is retry attempt {retries}. "
            "Your previous answer was incorrect or not grounded in the provided information. "
            "You MUST strictly follow all rules and not hallucinate.\n"
        )

    context_blocks = []
    if retrieved:
        context_blocks.append(f"[KNOWLEDGE BASE]\n{retrieved}")
    if tool_result:
        context_blocks.append(f"[TOOL RESULT]\n{tool_result}")

    context_section = "\n\n".join(context_blocks)

    prompt = f"""You are an intelligent customer support assistant for ShopMax (an e-commerce platform).

{name_part}
{retry_instruction}

SYSTEM ROLE:

You must correctly handle THREE types of user requests:

1. CHITCHAT
   - Greetings, small talk, casual conversation
   - Respond naturally, warmly, and briefly

2. TOOL-BASED
   - Calculations (discounts), date/time, utility queries
   - If a TOOL RESULT is provided → you MUST use it directly
   - Do NOT recompute or modify tool outputs

3. KNOWLEDGE-BASED (RAG)
   - Questions about ShopMax PRODUCTS or POLICIES


KNOWLEDGE STRUCTURE:

PRODUCTS:
- Product details such as features, specifications, price, discount, compatibility, warranty

POLICIES:
- Rules such as return policy, refund timelines, shipping details, payment methods, warranty terms


STRICT GROUNDING RULES:

When answering from KNOWLEDGE BASE:

1. Use ONLY the provided context
2. Do NOT use prior knowledge
3. Do NOT guess, infer, or assume missing details
4. Do NOT combine unrelated pieces of information
5. Extract exact facts (numbers, prices, durations, limits) precisely as written
6. If the answer is NOT explicitly present, respond EXACTLY:
   "I don't have that information in my knowledge base."


TOOL PRIORITY RULE:

If TOOL RESULT is present:
- Use it directly as the answer
- Do NOT add explanations, reasoning, or extra text
- Output only the final result in a natural sentence
- Do NOT override it with knowledge base information


ANSWER STYLE RULES:

- Be concise, clear, and helpful
- Do NOT repeat the question
- Do NOT mention "context", "documents", or internal reasoning
- Do NOT use emojis
- Address the user naturally if their name is available
-Do NOT add introductory phrases like:
"I have the information you need"
"Based on the information"
"Here is the answer"

Start directly with the answer.


DECISION LOGIC:

Follow this EXACT order:

Step 1: If TOOL RESULT exists → use it
Step 2: Else if question is about PRODUCTS or POLICIES → use KNOWLEDGE BASE
Step 3: Else → treat as CHITCHAT


CONTEXT:
{context_section if context_section else "No additional context available."}


CUSTOMER QUESTION:
{state['question']}

FINAL ANSWER:
"""

    answer = llm.invoke(prompt).content.strip()

    return {**state, 'answer': answer}


def eval_node(state: FinState) -> dict:
    if not state.get('retrieved'):
        return {**state, 'faithfulness': 1.0}
    llm = get_llm()
    prompt = (
        "Rate how faithful this answer is to the context.\n"
        "1.0 = only context facts used. 0.0 = hallucinated.\n\n"
        f"Context:\n{state['retrieved']}\n\nAnswer:\n{state['answer']}\n\n"
        "Reply with a single decimal 0.0-1.0. Nothing else."
    )
    raw = llm.invoke(prompt).content.strip()
    try:
        score = min(max(float(re.findall(r'\\d+\\.?\\d*', raw)[0]), 0.0), 1.0)
    except Exception:
        score = 0.5
    return {**state, 'faithfulness': score, 'eval_retries': state.get('eval_retries',0)+1}


def save_node(state: FinState) -> dict:
    msgs = (state.get('messages') or []) + [{'role':'assistant','content':state['answer']}]
    return {**state, 'messages': msgs[-6:]}


#GRAPH
@st.cache_resource
def build_graph():
    memory = MemorySaver()
    g = StateGraph(FinState)
    for name, fn in [('memory',memory_node),('router',router_node),('retrieve',retrieval_node),
                     ('skip',skip_node),('tool',tool_node),('answer',answer_node),
                     ('eval',eval_node),('save',save_node)]:
        g.add_node(name, fn)
    g.set_entry_point('memory')
    g.add_edge('memory','router')
    g.add_conditional_edges('router', lambda s: s['route'],
                            {'retrieve':'retrieve','tool':'tool','skip':'skip'})
    for n in ('retrieve','tool','skip'):
        g.add_edge(n,'answer')
    g.add_edge('answer','eval')
    g.add_conditional_edges('eval',
        lambda s: 'answer' if s['faithfulness'] < 0.7 and s['eval_retries'] < 2 else 'save',
        {'answer':'answer','save':'save'})
    g.add_edge('save', END)
    return g.compile(checkpointer=memory)


#SESSION STATE
if 'chat_history' not in st.session_state:
    st.session_state.chat_history = [
        {'role':'assistant','content':'Welcome to ShopMax Support. How can I help you today?'}
    ]
if 'thread_id' not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())
if 'trigger_query' not in st.session_state:
    st.session_state.trigger_query = None

graph = build_graph()


#  SIDEBAR
with st.sidebar:

    st.markdown("""
    <div class="sb-brand">
        <div class="sb-logo">S</div>
        <div>
            <div class="sb-title">ShopMax</div>
            <div class="sb-subtitle">Support Centre</div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown('<span class="sb-label">Search</span>', unsafe_allow_html=True)
    search_q = st.text_input(
        "search", placeholder="Search products or policies...",
        label_visibility="collapsed", key="search_bar"
    )
    if search_q:
        st.session_state.trigger_query = search_q

    st.divider()

    st.markdown('<span class="sb-label">Quick Actions</span>', unsafe_allow_html=True)
    c1, c2 = st.columns(2)
    with c1:
        if st.button("Track Order", use_container_width=True, key="qa_track"):
            st.session_state.trigger_query = "How do I track or cancel my order?"
    with c2:
        if st.button("View Offers", use_container_width=True, key="qa_offers"):
            st.session_state.trigger_query = "Which products have a discount right now?"
    if st.button("List All Products", use_container_width=True, key="qa_list"):
        st.session_state.trigger_query = "What products does ShopMax sell?"
    if st.button("Contact Support", use_container_width=True, key="qa_contact"):
        st.session_state.trigger_query = "How can I contact customer support?"

    st.divider()

    st.markdown('<span class="sb-label">Products</span>', unsafe_allow_html=True)
    with st.expander("Audio & Wearables"):
        if st.button("Wireless Headphones (WH-300)", key="p_wh"):
            st.session_state.trigger_query = "Tell me about Wireless Headphones WH-300"
        if st.button("True Wireless Earbuds (TWS-100)", key="p_tws"):
            st.session_state.trigger_query = "Tell me about True Wireless Earbuds TWS-100"
        if st.button("Blast Speaker (BS-50)", key="p_bs"):
            st.session_state.trigger_query = "Tell me about Portable Bluetooth Speaker BS-50"
        if st.button("Smart Watch (SW-200)", key="p_sw"):
            st.session_state.trigger_query = "Features of Smart Watch SW-200"

    with st.expander("Gaming & Accessories"):
        if st.button("Mechanical Keyboard (KB-75)", key="p_kb"):
            st.session_state.trigger_query = "Specs for Mechanical Keyboard KB-75"
        if st.button("Gaming Mouse (GM-80)", key="p_gm"):
            st.session_state.trigger_query = "Tell me about Gaming Mouse GM-80"
        if st.button("Laptop Stand (LS-20)", key="p_ls"):
            st.session_state.trigger_query = "Details on ErgoRise Laptop Stand LS-20"

    with st.expander("Storage & Power"):
        if st.button("External SSD 1TB", key="p_ssd"):
            st.session_state.trigger_query = "Tell me about External SSD SSD-1TB"
        if st.button("Power Bank 20000mAh", key="p_pb"):
            st.session_state.trigger_query = "Power Bank PB-20000 capacity"
        if st.button("65W GaN Charger (FC-65)", key="p_fc"):
            st.session_state.trigger_query = "Tell me about USB-C Fast Charger FC-65"

    st.divider()

    st.markdown('<span class="sb-label">Policies</span>', unsafe_allow_html=True)
    POLICIES = [
        ("Return Policy",                 "What is the return policy?"),
        ("Shipping Policy",               "How much is shipping?"),
        ("Refund Policy",                 "How long do refunds take?"),
        ("Warranty Policy",               "What is the warranty policy?"),
        ("Payment Methods",               "What payment methods do you accept?"),
        ("Order Tracking & Cancellation", "How do I track or cancel an order?"),
        ("Exchange Policy",               "What is the exchange policy?"),
        ("Privacy Policy",                "What is the privacy policy?"),
    ]
    for label, query in POLICIES:
        if st.button(label, key=f"pol_{label}"):
            st.session_state.trigger_query = query

    st.divider()

    st.markdown('<span class="sb-label">Session</span>', unsafe_allow_html=True)
    if st.button("Clear Conversation", use_container_width=True, key="clear_btn"):
        st.session_state.chat_history = [
            {'role':'assistant','content':'Conversation cleared. How can I help you?'}
        ]
        st.session_state.thread_id = str(uuid.uuid4())
        st.rerun()


#  MAIN CHAT AREA
st.markdown("""
<div class="main-header">
    <div>
        <div class="main-title">ShopMax Support</div>
        <div class="main-subtitle">Intelligent assistance &middot; Products &middot; Policies &middot; Orders</div>
    </div>
    <div class="status-pill"><div class="status-dot"></div>LIVE</div>
</div>
""", unsafe_allow_html=True)

st.markdown("""
<div class="info-strip">
    <div class="info-chip">Mon&ndash;Sat <b>9am&ndash;7pm IST</b></div>
    <div class="info-chip">Email <b>support@shopmax.in</b></div>
    <div class="info-chip">Toll-free <b>1800-123-4567</b></div>
    <div class="info-chip">Reply <b>&lt;24h</b></div>
</div>
""", unsafe_allow_html=True)

# Render history
for msg in st.session_state.chat_history:
    role_class = "user" if msg["role"] == "user" else "bot"

    with st.chat_message(msg["role"]):
        st.markdown(
            f"""
            <div class="chat-row {role_class}">
                <div class="chat-bubble">
                    {msg["content"]}
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

# Single chat_input call
user_input = st.chat_input("Ask about products, returns, shipping, or anything else...")

# Handle sidebar triggers
if st.session_state.trigger_query:
    user_input = st.session_state.trigger_query
    st.session_state.trigger_query = None

# Process
if user_input:
    with st.chat_message('user'):
        st.markdown(
            f"""
            <div class="chat-row user">
                <div class="chat-bubble">
                    {user_input}
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    state = {
        'question':     user_input,
        'messages':     st.session_state.chat_history,
        'route':        '',
        'retrieved':    '',
        'sources':      [],
        'tool_result':  '',
        'answer':       '',
        'faithfulness': 1.0,
        'eval_retries': 0,
        'user_name':    None,
    }
    config = {'configurable': {'thread_id': st.session_state.thread_id}}

    with st.chat_message('assistant'):
        ph = st.empty()

        ph.markdown("""
        <div class="chat-row bot">
            <div class="chat-bubble">
                <div class="typing-wrap">
                    <div class="tdot"></div>
                    <div class="tdot"></div>
                    <div class="tdot"></div>
                </div>
            </div>
        </div>
        """, unsafe_allow_html=True)

        result = graph.invoke(state, config)

        ph.markdown(
            f"""
            <div class="chat-row bot">
                <div class="chat-bubble">
                    {result['answer']}
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    st.session_state.chat_history = result['messages']


'''
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_CODE)
print('app.py written successfully.')

app.py written successfully.


## Part 7 — Launch Streamlit via ngrok

In [40]:
from pyngrok import ngrok, conf

if not NGROK_TOKEN:
    print('ERROR: NGROK_AUTH_TOKEN not found in Colab Secrets. Cannot launch UI.')
else:
    conf.get_default().auth_token = NGROK_TOKEN

    ngrok.kill()
    subprocess.run(['pkill', '-f', 'streamlit'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)

    subprocess.Popen(
        ['streamlit', 'run', 'app.py',
         '--server.port', '8501',
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(5)

    public_url = ngrok.connect(8501)
    print('\nShopMax Support Bot is live at:\n')
    print(public_url.public_url)
    print('\nOpen the URL above in your browser.')


ShopMax Support Bot is live at:

https://tribune-zone-unrevised.ngrok-free.dev

Open the URL above in your browser.


In [38]:
!pkill -f streamlit
!pkill ngrok
print('Server stopped.')

Server stopped.
